In [ ]:
import cv2
from google.colab.patches import cv2_imshow

In [ ]:
img_filename = "./images/{0}_00147.{1}"
img = cv2.imread(img_filename.format("rgb", "jpg"))
cv2_imshow(img)

AttributeError: 'NoneType' object has no attribute 'clip'

In [ ]:
print(img)

None


In [ ]:
depth = cv2.imread(img_filename.format("depth", "png"), cv2.IMREAD_ANYCOLOR | cv2.IMREAD_ANYDEPTH)
depth

In [ ]:
# # IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# # THEN FEEL FREE TO DELETE THIS CELL.
# # NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# # ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# # NOTEBOOK.
# import kagglehub
# mikoajkoek_traffic_road_object_detection_polish_12k_path = kagglehub.dataset_download('mikoajkoek/traffic-road-object-detection-polish-12k')

# print('Data source import complete.')


: 

In [ ]:
from transformers import pipeline
import torch
from accelerate.test_utils.testing import get_backend
from PIL import Image
device, _, _ = get_backend()
checkpoint = "Intel/zoedepth-kitti"
pipe = pipeline("depth-estimation", model=checkpoint, device=device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/2.12k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.38G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/553 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

In [ ]:
!pip install ultralytics

# from google.colab import drive
# drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.1 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/bhaskrr/traffic-sign-detection-using-yolov11.git

: 

In [ ]:
!pip install -r ./traffic-sign-detection-using-yolov11/requirements.txt

: 

In [ ]:
# load the sign detection model
from ultralytics import YOLO

# initialize the detector with weights
detector = YOLO("yolo11n.pt")
detections = detector.predict(img)

print("detections:", len(detections))
for detection in detections:
    # Get class indices and class names
    class_ids = detection.boxes.cls  # cls stores the class IDs
    print(class_ids)
    # Iterate over bounding boxes
    for i, bbox in enumerate(detection.boxes):
        # Get the bounding box coordinates
        x1, y1, x2, y2 = bbox.xyxy[0]
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        # Draw the bounding box
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)

        # Get class name using the class ID
        class_id = int(class_ids[i])
        class_name = detection.names[class_id]  # detection.names holds the class names

        # Display the class name on the image
        cv2.putText(img, class_name, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

        # Show the image with detection
        cv2_imshow(img)

        # # Break on 'q' key
        # if cv2_imshow.waitKey(10000) & 0xFF == ord("q"):
        #     break

: 

In [ ]:
import os
import cv2
from google.colab.patches import cv2_imshow
import numpy as np


img_width, img_height = img.shape[0], img.shape[1]

# Helper function to parse a single YOLO label line
def parse_yolo_label(label_line):
    label_parts = label_line.strip().split()
    class_id = int(label_parts[0])
    center_x = float(label_parts[1])
    center_y = float(label_parts[2])
    width = float(label_parts[3])
    height = float(label_parts[4])
    return class_id, center_x, center_y, width, height

# Convert YOLO format to bounding box coordinates in image pixel values
def yolo_to_image_coords(center_x, center_y, width, height, img_width, img_height):
    box_x1 = int((center_x - width / 2) * img_width)
    box_y1 = int((center_y - height / 2) * img_height)
    box_x2 = int((center_x + width / 2) * img_width)
    box_y2 = int((center_y + height / 2) * img_height)
    return box_x1, box_y1, box_x2, box_y2

# Load images and corresponding labels
def load_data(images_path, labels_path):
    images = []
    boxes = []
    class_ids = []

    for image_file in os.listdir(images_path):
        if image_file.endswith('.jpg') or image_file.endswith('.png'):
            # Load image
            image_path = os.path.join(images_path, image_file)
            image = cv2.imread(image_path)
            image = cv2.resize(image, (img_width, img_height))  # Adjust size

            # Append image to list
            images.append(image)

            # Load corresponding label
            label_file = os.path.splitext(image_file)[0] + '.txt'
            label_path = os.path.join(labels_path, label_file)

            with open(label_path, 'r') as file:
                label_data = file.readlines()
                image_boxes = []
                image_class_ids = []

                for label_line in label_data:
                    class_id, center_x, center_y, width, height = parse_yolo_label(label_line)
                    box_x1, box_y1, box_x2, box_y2 = yolo_to_image_coords(center_x, center_y, width, height, img_width, img_height)

                    # Append box and class_id to respective lists
                    image_boxes.append([box_x1, box_y1, box_x2, box_y2])
                    image_class_ids.append(class_id)

                boxes.append(image_boxes)
                class_ids.append(image_class_ids)

    return np.array(images), boxes, class_ids

# Visualize an image with bounding boxes drawn
def visualize_image_with_boxes(image, boxes, class_ids):
    for (box, class_id) in zip(boxes, class_ids):
        x1, y1, x2, y2 = box
        # Draw rectangle and class id
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(image, f'Class {class_id}', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (36,255,12), 2)

    # cv2.imshow("Image with Bounding Boxes", image)
    cv2_imshow(image)



: 

In [ ]:
est_depth = pipe(img_filename.format("rgb", "jpg"))["depth"]

: 

In [ ]:
est_depth

: 

In [ ]:
est_depth_np = np.array(est_depth)
est_depth_np

: 

In [ ]:
depth

: 

In [ ]:
est_depth_np/depth

: 

: 

In [ ]:
def visualize_boxes(i):
  visualize_image_with_boxes(train_images[i].copy(), train_boxes[i], train_class_ids[i])

: 

In [ ]:
def visualize_image_with_sign_boxes(image, boxes, class_ids):
    for (box, class_id) in zip(boxes, class_ids):
      if class_id in sign_ids:
        x1, y1, x2, y2 = box
        # Draw rectangle and class id
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(image, f'Class {class_id}', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (36,255,12), 2)

    # cv2.imshow("Image with Bounding Boxes", image)
    cv2_imshow(image)


def visualize_sign_boxes(i):
  visualize_image_with_sign_boxes(train_images[i].copy(), train_boxes[i], train_class_ids[i])

: 

In [ ]:
i = np.random.randint(484)
visualize_boxes(i)
visualize_sign_boxes(i)

: 

In [ ]:
pil_train_images = [Image.fromarray(image) for image in train_images]
predictions = pipe(pil_train_images[0])
predictions["depth"]

: 

In [ ]:
def visualize_pred_crop(i, j):
  box=sign_boxes[i][j]
  return Image.fromarray(np.array(pipe(Image.fromarray(train_images[i]))["depth"])[box[1]:box[3],box[0]:box[2]])

def visualize_crop(i, j):
  box=sign_boxes[i][j]
  return Image.fromarray(np.array(train_images[i])[box[1]:box[3],box[0]:box[2]])

: 

In [ ]:
def visualize_pred(i):
  return pipe(Image.fromarray(train_images[i]))["depth"]
visualize_pred(0)

: 

In [ ]:
pred_raw = [pipe(image)["predicted_depth"] for image in pil_train_images]

: 

In [ ]:
def pred_sign_raw(i, j):
  box=sign_boxes[i][j]
  image=pred_raw[i]
  return image[box[1]:box[3],box[0]:box[2]]
def pred_sign_dist(i, j):
  return torch.Tensor.mean(pred_sign_raw(i, j))

: 

In [ ]:
sign_distances = []
for i in range(len(sign_boxes)):
  sign_distances.append([])
  for j in range(len(sign_boxes[i])):
    sign_distances[i].append(pred_sign_dist(i,j))

: 

In [ ]:
print(sign_distances)

: 

In [ ]:
def test_dist(i):
  print(sign_distances[i])
  visualize_sign_boxes(i)

: 

In [ ]:
test_dist(np.random.randint(len(sign_boxes)))

: 